# Install Dependencies

In [1]:
!pip install -q arxiv transformers sentence-transformers pandas scikit-learn streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 112.5 MB/s eta 0:00:00


# Imports

In [9]:
!pip install transformers torch

In [1]:
import arxiv
import pandas as pd


from transformers import pipeline


# Fetch Papers from arXiv (FIXED API)

In [21]:
query = "artificial intelligence"



client = arxiv.Client(
    page_size=100,     # fetch 100 per request (max allowed by arXiv API)
    delay_seconds=3,   # polite delay to avoid rate-limiting
    num_retries=5      # retry on transient failures
)

papers = []

for result in client.results(client):
    papers.append({
        "title": result.title,
        "abstract": result.summary,
        "authors": ", ".join([a.name for a in result.authors]),
        "published": result.published,
        "pdf": result.pdf_url
    })

df = pd.DataFrame(papers)

df.head()

,title,abstract,authors,published,pdf
0,Lumos-Nexus: Efficient Frequency Bridging with...,Connector-based video unified models have demo...,"Jiazheng Xing, Hangjie Yuan, Lingling Cai, Xin...",2026-05-29 17:59:50+00:00,https://arxiv.org/pdf/2605.31603v1
1,Stateful Online Monitoring Catches Distributed...,Language models can find thousands of severe s...,"Davis Brown, Samarth Bhargav, Arav Santhanam, ...",2026-05-29 17:57:00+00:00,https://arxiv.org/pdf/2605.31593v1
2,TunerDiT: Training-free Progressive Steering o...,Text-to-video (T2V) generation faces challengi...,"Ruotong Liao, Guowen Huang, Qing Cheng, Guangy...",2026-05-29 17:56:09+00:00,https://arxiv.org/pdf/2605.31590v1
3,Language Models Learn Constructional Semantics...,Grasping the semantics of rare constructions (...,"Wesley Scivetti, Ethan Wilcox, Nathan Schneide...",2026-05-29 17:54:00+00:00,https://arxiv.org/pdf/2605.31586v1
4,LongTraceRL: Learning Long-Context Reasoning f...,Long-context reasoning remains a central chall...,"Nianyi Lin, Jiajie Zhang, Lei Hou, Juanzi Li",2026-05-29 17:51:40+00:00,https://arxiv.org/pdf/2605.31584v1


In [22]:
print(f"Total papers fetched: {len(df)}")

Total papers fetched: 200


# Load Summarizer

In [3]:
from transformers import pipeline

In [15]:
!pip uninstall -y transformers
!pip install transformers==4.41.1 accelerate torch sentencepiece

Found existing installation: transformers 4.41.1
Uninstalling transformers-4.41.1:
  Successfully uninstalled transformers-4.41.1
  Using cached transformers-4.41.1-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.41.1-py3-none-any.whl (9.1 MB)


In [4]:
import transformers
print(transformers.__version__)

4.41.1


In [23]:
summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6"
)

# Create Safe Summarization Function

In [24]:
def summarize(text):
    try:
        text = str(text)
        text = text.replace("\n", " ")[:800]

        result = summarizer(text, max_length=120, min_length=30, do_sample=False)

        print("DEBUG OUTPUT:", result)

        return result[0]["summary_text"]

    except Exception as e:
        print("ERROR:", e)
        return "Summary failed"

In [25]:
summarize(df["abstract"].iloc[0])

DEBUG OUTPUT: [{'summary_text': ' Connector-based video unified models have demonstrated strong capability in instruction-grounded video synthesis . We propose Lumos-Nexus, a training-efficient unified video generation framework that facilitates the development of strong reasoning-driven generation capabilities while enhancing visual fidelity .'}]


' Connector-based video unified models have demonstrated strong capability in instruction-grounded video synthesis . We propose Lumos-Nexus, a training-efficient unified video generation framework that facilitates the development of strong reasoning-driven generation capabilities while enhancing visual fidelity .'

# Apply Summarization on DataFrame

In [26]:
from tqdm import tqdm

tqdm.pandas()

df["summary_generated"] = df["abstract"].progress_apply(summarize)

  1%|          | 2/200 [00:08<13:12,  4.00s/it]

DEBUG OUTPUT: [{'summary_text': ' Connector-based video unified models have demonstrated strong capability in instruction-grounded video synthesis . We propose Lumos-Nexus, a training-efficient unified video generation framework that facilitates the development of strong reasoning-driven generation capabilities while enhancing visual fidelity .'}]


  2%|▏         | 3/200 [00:19<22:53,  6.97s/it]

DEBUG OUTPUT: [{'summary_text': ' Language models can find thousands of severe software vulnerabilities, and agents are increasingly being misused for cyberattacks . To avoid detection, attackers frequently distribute their misuse, splitting a harmful task across many user accounts . Safety monitors score only one agent context at a time, they are structurally blind to misuse .'}]


  2%|▏         | 4/200 [00:34<33:23, 10.22s/it]

...

DEBUG OUTPUT: [{'summary_text': ' U.S. immigration law spans thousands of pages of official policy, federal regulations, and procedural guidance that change frequently . We describe the construction of ImmigrationQA, a source-grounded question-answering dataset of 17,058 pairs across 13 immigration subdomains .'}]


100%|██████████| 200/200 [33:03<00:00,  9.52s/it]

DEBUG OUTPUT: [{'summary_text': ' Effective prognostics and health management of modern engines relies on accurate turbine gas temperature predictions and robust uncertainty quantification . This paper investigates five major approaches for constructing prediction intervals . Each approach is implemented within a unified experimental framework .'}]


100%|██████████| 200/200 [33:12<00:00,  9.96s/it]

DEBUG OUTPUT: [{'summary_text': ' Industrial visual sim-to-real is often described as transferring from synthetic images to real images . Industrial deployment usually involves a mismatch between available evidence and required decisions . We distinguish CAD-available settings, where explicit object geometry can support rendering, .'}]


# Clean Final Table

In [27]:
df_final = df[["title", "authors", "published", "pdf", "summary_generated"]]
df_final.head()

,title,authors,published,pdf,summary_generated
0,Lumos-Nexus: Efficient Frequency Bridging with...,"Jiazheng Xing, Hangjie Yuan, Lingling Cai, Xin...",2026-05-29 17:59:50+00:00,https://arxiv.org/pdf/2605.31603v1,Connector-based video unified models have dem...
1,Stateful Online Monitoring Catches Distributed...,"Davis Brown, Samarth Bhargav, Arav Santhanam, ...",2026-05-29 17:57:00+00:00,https://arxiv.org/pdf/2605.31593v1,Language models can find thousands of severe ...
2,TunerDiT: Training-free Progressive Steering o...,"Ruotong Liao, Guowen Huang, Qing Cheng, Guangy...",2026-05-29 17:56:09+00:00,https://arxiv.org/pdf/2605.31590v1,Text-to-video (T2V) generation faces challeng...
3,Language Models Learn Constructional Semantics...,"Wesley Scivetti, Ethan Wilcox, Nathan Schneide...",2026-05-29 17:54:00+00:00,https://arxiv.org/pdf/2605.31586v1,Grasping the semantics of rare constructions ...
4,LongTraceRL: Learning Long-Context Reasoning f...,"Nianyi Lin, Jiajie Zhang, Lei Hou, Juanzi Li",2026-05-29 17:51:40+00:00,https://arxiv.org/pdf/2605.31584v1,Long-context reasoning remains a central chal...


# Sort Papers

In [28]:
df_final = df_final.sort_values(by="published", ascending=False)

# Save Output

In [30]:
df_final.to_csv("arxiv_summaries.csv", index=False)